Step 1: Deciding on Data Sources
As a group, decide on a theme (or related themes) that you want to explore as your final project topic. It should be a topic that is of relevance to you (e.g. a hobby you are very passionate about) or the world (e.g. climate change, food insecurity). Ideally, your topic should surround a modern local, national or global issue. Determine a handful of datasets that could be more rich when considered together. A good start would be to choose a "core" dataset and then finding 1-2 supporting sources. If you have an idea for a topic but can't find supporting data, ask Dr. Mendible for help!

Individually, each group member should download and investigate one unique data set. Answer the following questions about each one:

What is available in this data set? What are the variables (either a complete list or a summary of the theme)? How many observations are included?
Are there any challenges with this data set? Does it need to be cleaned, manipulated, or reshaped to be useful? Are these manipulations straightforward or will they take longer?
How will this data be merged or used with the rest of the data?
Then, share in your groups what you found. Make a team decision on what data you will be using for the final project.

In [1]:
!pip install vega_datasets
!pip install pycountry

    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python310\python.exe -m pip install --upgrade pip


    tinycss2 (>=1.1.0<1.2) ; extra == 'css'
             ~~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python310\python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import altair as alt
from vega_datasets import data as vega_data
import pycountry

import os
import requests

In [12]:
energy_consumption_by_source_and_country_filename = "energy-consumption-by-source-and-country.csv"
if not os.path.exists(energy_consumption_by_source_and_country_filename):
  resp = requests.get("https://raw.githubusercontent.com/roy-y-2023/data3310/refs/heads/main/data/energy-consumption-by-source-and-country.csv")
  with open(energy_consumption_by_source_and_country_filename, "wb") as f:
    f.write(resp.content)
energy_consumption_by_source_and_country_df = pd.read_csv(energy_consumption_by_source_and_country_filename)
energy_consumption_by_source_and_country_df.head()

,Entity,Code,Year,Other renewables,Biofuels,Solar,Wind,Hydropower,Nuclear,Gas,Coal,Oil
0,Africa,NaN,1965,0.0,0.0,0.0,0.0,38.626762,0.0,9.571930,323.49615,341.07257
1,Africa,NaN,1966,0.0,0.0,0.0,0.0,43.083344,0.0,10.698092,323.12222,369.46457
2,Africa,NaN,1967,0.0,0.0,0.0,0.0,44.973988,0.0,10.573844,330.29160,368.28156
3,Africa,NaN,1968,0.0,0.0,0.0,0.0,52.606503,0.0,10.717145,343.51288,389.57162
4,Africa,NaN,1969,0.0,0.0,0.0,0.0,61.391357,0.0,12.520175,346.64288,397.37260


In [13]:
def get_numeric_code(row):
  if pd.isna(row["Code"]):
    return None
  country = pycountry.countries.get(alpha_3=row["Code"])
  if country:
    return int(country.numeric)
  return None
energy_consumption_by_source_and_country_df["Code_Num"] = energy_consumption_by_source_and_country_df.apply(get_numeric_code, axis=1)

energy_consumption_by_source_and_country_df["Total_Conventional_Energy"] = energy_consumption_by_source_and_country_df["Gas"] + energy_consumption_by_source_and_country_df["Coal"] + energy_consumption_by_source_and_country_df["Oil"]
energy_consumption_by_source_and_country_df["Total_Clean_Energy"] = energy_consumption_by_source_and_country_df["Other renewables"] + energy_consumption_by_source_and_country_df["Biofuels"] + energy_consumption_by_source_and_country_df["Solar"] + energy_consumption_by_source_and_country_df["Wind"] + energy_consumption_by_source_and_country_df["Hydropower"] + energy_consumption_by_source_and_country_df["Nuclear"]
energy_consumption_by_source_and_country_df["Total"] = energy_consumption_by_source_and_country_df["Total_Conventional_Energy"] + energy_consumption_by_source_and_country_df["Total_Clean_Energy"]
energy_consumption_by_source_and_country_df["Clean_Energy_Share"] = energy_consumption_by_source_and_country_df["Total_Clean_Energy"] / energy_consumption_by_source_and_country_df["Total"] * 100

energy_consumption_by_source_and_country_df = energy_consumption_by_source_and_country_df[energy_consumption_by_source_and_country_df["Year"] == 2024]

energy_consumption_by_source_and_country_df.head()

,Entity,Code,Year,Other renewables,Biofuels,Solar,Wind,Hydropower,Nuclear,Gas,Coal,Oil,Code_Num,Total_Conventional_Energy,Total_Clean_Energy,Total,Clean_Energy_Share
59,Africa,NaN,2024,33.729378,1.159756,50.888393,70.906006,414.818970,19.024391,1780.34900,1194.997100,2543.77340,NaN,5519.119500,590.526894,6109.646394,9.665484
119,Africa (EI),NaN,2024,33.729378,1.159756,50.888393,70.906006,414.818970,19.024391,1780.34900,1194.997100,2543.77340,NaN,5519.119500,590.526894,6109.646394,9.665484
179,Algeria,DZA,2024,0.000000,0.000000,1.672863,0.039131,0.124829,0.000000,505.47590,1.736987,250.97054,12.0,758.183427,1.836824,760.020251,0.241681
239,Angola,AGO,2024,NaN,0.000000,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,24.0,NaN,NaN,NaN,NaN
299,Argentina,ARG,2024,8.232407,14.633476,9.671616,39.432907,72.362910,25.485405,456.22565,9.952372,339.03214,32.0,805.210162,169.818721,975.028883,17.416789


In [14]:
energy_consumption_by_source_and_country_df[energy_consumption_by_source_and_country_df["Clean_Energy_Share"].isna()]

,Entity,Code,Year,Other renewables,Biofuels,Solar,Wind,Hydropower,Nuclear,Gas,Coal,Oil,Code_Num,Total_Conventional_Energy,Total_Clean_Energy,Total,Clean_Energy_Share
239,Angola,AGO,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,24.0,NaN,NaN,NaN,NaN
639,Bahrain,BHR,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,48.0,NaN,NaN,NaN,NaN
854,Bolivia,BOL,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,68.0,NaN,NaN,NaN,NaN
974,Brunei,BRN,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,96.0,NaN,NaN,NaN,NaN
1274,Chad,TCD,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,148.0,NaN,NaN,NaN,NaN
1514,Congo,COG,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,178.0,NaN,NaN,NaN,NaN
1579,Cuba,CUB,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,192.0,NaN,NaN,NaN,NaN
1639,Curacao,CUW,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,531.0,NaN,NaN,NaN,NaN
1789,Democratic Republic of Congo,COD,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,180.0,NaN,NaN,NaN,NaN
2089,Equatorial Guinea,GNQ,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,226.0,NaN,NaN,NaN,NaN


In [17]:
# Patch NaN Clean_Energy_Share for real countries using their continent/region row values (from dataset)
# Non-country entities (Low-income countries, Non-OPEC (EI), OPEC (EI), Other * (EI), Rest of World (EI)) are excluded

def region_value(region_name):
    return energy_consumption_by_source_and_country_df.loc[
        energy_consumption_by_source_and_country_df["Entity"] == region_name, "Clean_Energy_Share"
    ].values[0]

country_to_region = {
    # Middle Africa
    "Congo":                        "Middle Africa (EI)",
    "Democratic Republic of Congo": "Middle Africa (EI)",
    "Equatorial Guinea":            "Middle Africa (EI)",
    "Gabon":                        "Middle Africa (EI)",
    # Eastern Africa
    "Madagascar":                   "Eastern Africa (EI)",
    "Mozambique":                   "Eastern Africa (EI)",
    "South Sudan":                  "Eastern Africa (EI)",
    "Sudan":                        "Eastern Africa (EI)",
    "Zambia":                       "Eastern Africa (EI)",
    "Zimbabwe":                     "Eastern Africa (EI)",
    # Western Africa
    "Nigeria":                      "Western Africa (EI)",
    # Northern Africa
    "Tunisia":                      "Other Northern Africa (EI)",
    "Libya":                        "Other Northern Africa (EI)",
    # Caribbean
    "Cuba":                         "Other Caribbean (EI)",
    "Curacao":                      "Other Caribbean (EI)",
    "Netherlands Antilles":         "Other Caribbean (EI)",
    # South America
    "Guyana":                       "South America",
    # Asia
    "Mongolia":                     "Asia",
    "Myanmar":                      "Asia",
    # Oceania
    "New Caledonia":                "Oceania",
    "Papua New Guinea":             "Oceania",
    # Europe
    "Serbia":                       "Europe",
    # Middle East
    "Syria":                        "Middle East (EI)",
    "Yemen":                        "Middle East (EI)",
}

for country, region in country_to_region.items():
    energy_consumption_by_source_and_country_df.loc[
        energy_consumption_by_source_and_country_df["Entity"] == country, "Clean_Energy_Share"
    ] = region_value(region)

energy_consumption_by_source_and_country_df[energy_consumption_by_source_and_country_df["Clean_Energy_Share"].isna()]

,Entity,Code,Year,Other renewables,Biofuels,Solar,Wind,Hydropower,Nuclear,Gas,Coal,Oil,Code_Num,Total_Conventional_Energy,Total_Clean_Energy,Total,Clean_Energy_Share,Continent
239,Angola,AGO,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,24.0,NaN,NaN,NaN,NaN,NaN
639,Bahrain,BHR,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,48.0,NaN,NaN,NaN,NaN,NaN
854,Bolivia,BOL,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,68.0,NaN,NaN,NaN,NaN,NaN
974,Brunei,BRN,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,96.0,NaN,NaN,NaN,NaN,NaN
1274,Chad,TCD,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,148.0,NaN,NaN,NaN,NaN,NaN
3689,Low-income countries,NaN,2024,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Africa
4658,Non-OPEC (EI),NaN,2024,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Asia
4993,OPEC (EI),NaN,2024,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Asia
5173,Other Africa (EI),NaN,2024,NaN,0.136209,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Africa
5397,Other Eastern Africa (EI),NaN,2024,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Africa


Step 2: Visualization Brainstorming

Then as a group, brainstorm a list of at least 5 visualizations that could be made with this collective information. Come up with a one-sentence statement or question that the graphic would address. For each of the visualizations, answer the following questions. Your answer can be in the form of a mock-up if you have ready-to-plot data.

What variables would you use to show this?
Do you need to compute new variables, filter, or do other manipulations to show this?
What chart type (line, scatter, bar, etc) would be most effective to show this?
Which encodings would you map to which variables?
What other design considerations (color, scales, titles, labels, ticks) would you need to make? If you don't know yet, that's ok, but give a best guess.
Compile your answers into a single notebook and submit it here.

Visualization 1:

A plot representing the relationship between

Visualization 2:

Visualization 3:

I'm using the share of clean energy for a given country as percentage, computed from the raw values, that's neccessary for showing on Choropleths and other types of charts in general as different country have different baseline that will cause many smaller countries to have nearly zero.

Alternatively it can be shown as bar chart. Although the Choropleths chart isn't good at showing precise values, but it does show off geographical relationship of these values, that countries next to each other might have similar share of clean energy, and some regions have high share of clean energy.



In [18]:
countries = alt.topo_feature(vega_data.world_110m.url, 'countries')

alt.Chart(countries).mark_geoshape().encode(
    alt.Color("Clean_Energy_Share:Q", bin=True),
    tooltip='Entity:N'
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(energy_consumption_by_source_and_country_df, 'Code_Num', ["Clean_Energy_Share","Entity"])
).project(
    "naturalEarth1"
).properties(
    title='Share of Clean Energy (2024)',
    width=600,
    height=400
)

alt.Chart(...)

Visualization 4:

Visualization 5: